SCENARIO: “Banking Smart Assistant System”
🏦 Background Story
A major bank launches an AI-powered customer assistant.
👉 Customers can ask:
- “What is my account balance?”
- “Show me my last 5 transactions.”
- “When is my loan EMI due?”
👉 Instead of logging into apps or waiting on customer service calls,
👉 AI fetches the information instantly, securely, and in real time.

⚙️ Core Idea:
Just like the hospital and college scenarios, the assistant removes manual checking, centralizes financial data, and makes access instant.
💡 Impact:
- Saves customers time.
- Reduces load on call centers.
- Provides personalized financial insights on demand.

In [ ]:
# ======================================
# STEP 1: Install Libraries
# ======================================
!pip install groq gradio nest_asyncio


# ======================================
# STEP 2: Load API Key using os.environ
# ======================================
import os

# Replace with your NEW valid Groq API key
os.environ["GROQ_API_KEY"] = "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"

from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))


# ======================================
# STEP 3: MOCK TOOLS (Banking MCP Tools)
# ======================================
import asyncio
import nest_asyncio

# Keeping structure same
nest_asyncio.apply()

# --------------------------------------
# Compatibility patch for Gradio/Uvicorn
# --------------------------------------
_original_asyncio_run = asyncio.run

def compatible_asyncio_run(main, *, debug=None, loop_factory=None):
    return _original_asyncio_run(main)

asyncio.run = compatible_asyncio_run


customers = {
    "C101": {
        "name": "Rahul Sharma",
        "account_type": "Savings",
        "account_balance": "₹85,450",
        "last_transactions": [
            {"date": "2026-03-29", "type": "Debit", "amount": "₹1,250", "description": "Electricity Bill"},
            {"date": "2026-03-28", "type": "Credit", "amount": "₹25,000", "description": "Salary Credit"},
            {"date": "2026-03-27", "type": "Debit", "amount": "₹799", "description": "Online Shopping"},
            {"date": "2026-03-26", "type": "Debit", "amount": "₹2,100", "description": "Restaurant Payment"},
            {"date": "2026-03-25", "type": "Debit", "amount": "₹500", "description": "Mobile Recharge"}
        ],
        "loan_info": {
            "loan_type": "Home Loan",
            "emi_amount": "₹18,500",
            "emi_due_date": "2026-04-05",
            "status": "Upcoming"
        }
    },
    "C102": {
        "name": "Priya Verma",
        "account_type": "Current",
        "account_balance": "₹1,42,300",
        "last_transactions": [
            {"date": "2026-03-29", "type": "Debit", "amount": "₹5,000", "description": "ATM Withdrawal"},
            {"date": "2026-03-28", "type": "Debit", "amount": "₹2,450", "description": "Grocery Store"},
            {"date": "2026-03-27", "type": "Credit", "amount": "₹12,000", "description": "Client Payment"},
            {"date": "2026-03-26", "type": "Debit", "amount": "₹1,200", "description": "Fuel Payment"},
            {"date": "2026-03-25", "type": "Debit", "amount": "₹950", "description": "Internet Bill"}
        ],
        "loan_info": {
            "loan_type": "Car Loan",
            "emi_amount": "₹12,200",
            "emi_due_date": "2026-04-10",
            "status": "Upcoming"
        }
    }
}


async def get_account_balance(customer_id):
    await asyncio.sleep(1)
    if customer_id in customers:
        customer = customers[customer_id]
        return (
            f"🏦 Account Balance Details:\n"
            f"- Customer Name: {customer['name']}\n"
            f"- Account Type: {customer['account_type']}\n"
            f"- Available Balance: {customer['account_balance']}"
        )
    return "❌ Customer not found."


async def get_last_transactions(customer_id):
    await asyncio.sleep(1)
    if customer_id in customers:
        transactions = customers[customer_id]["last_transactions"]
        if not transactions:
            return "💳 No recent transactions found."

        text = "💳 Last 5 Transactions:\n"
        for tx in transactions:
            text += (
                f"- Date: {tx['date']}, Type: {tx['type']}, "
                f"Amount: {tx['amount']}, Description: {tx['description']}\n"
            )
        return text.strip()
    return "❌ Customer not found."


async def get_loan_emi_details(customer_id):
    await asyncio.sleep(1)
    if customer_id in customers:
        loan = customers[customer_id]["loan_info"]
        return (
            f"📅 Loan EMI Details:\n"
            f"- Loan Type: {loan['loan_type']}\n"
            f"- EMI Amount: {loan['emi_amount']}\n"
            f"- EMI Due Date: {loan['emi_due_date']}\n"
            f"- Status: {loan['status']}"
        )
    return "❌ Customer not found."


# ======================================
# STEP 4: PARALLEL TOOL INVOCATION
# ======================================
async def parallel_customer_research(customer_id):
    results = await asyncio.gather(
        get_account_balance(customer_id),
        get_last_transactions(customer_id),
        get_loan_emi_details(customer_id),
        return_exceptions=True
    )

    balance, transactions, loan = results

    return {
        "balance": balance if not isinstance(balance, Exception) else "Balance data unavailable",
        "transactions": transactions if not isinstance(transactions, Exception) else "Transaction data unavailable",
        "loan": loan if not isinstance(loan, Exception) else "Loan data unavailable"
    }


# ======================================
# STEP 5: CHAINED TOOL INVOCATION USING GROQ
# ======================================
def decide_intent(user_query):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
You are a banking smart assistant intent classifier.

Classify the user's query into exactly one of these:
- balance
- transactions
- loan_emi
- full_summary

Rules:
- If the user asks about account balance, available amount, savings balance -> balance
- If the user asks about statement, transactions, spending history, recent payments -> transactions
- If the user asks about EMI, loan due date, loan payment -> loan_emi
- If the user asks for complete banking summary, all details, full account overview -> full_summary

Only return one label.

User Query: {user_query}
"""
        }]
    )
    return response.choices[0].message.content.strip().lower()


def analyse_banking_data(text):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Analyze this banking data and provide:
1. Key financial summary
2. Important observations
3. Spending or transaction note
4. Loan/EMI reminder if any
5. Simple customer-friendly explanation

Data:
{text}
"""
        }]
    )
    return response.choices[0].message.content


def generate_banking_report(analysis, customer_id):
    customer_name = customers.get(customer_id, {}).get("name", "Customer")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Create a professional but simple banking report for {customer_name}.

Use this analysis:
{analysis}

Keep it:
- clear
- structured
- customer-friendly
- concise but informative
"""
        }]
    )
    return response.choices[0].message.content


# ======================================
# STEP 6: FULL MCP PIPELINE
# ======================================
async def full_pipeline(customer_id, user_query):
    if customer_id not in customers:
        return "❌ Invalid Customer ID. Please enter C101 or C102."

    intent = decide_intent(user_query)
    data = await parallel_customer_research(customer_id)

    if intent == "balance":
        combined_text = data["balance"]

    elif intent == "transactions":
        combined_text = data["transactions"]

    elif intent == "loan_emi":
        combined_text = data["loan"]

    else:
        combined_text = f"""
{data['balance']}

{data['transactions']}

{data['loan']}
""".strip()

    analysis = analyse_banking_data(combined_text)
    report = generate_banking_report(analysis, customer_id)

    final_output = f"""
==============================
BANKING SMART ASSISTANT OUTPUT
==============================

Customer ID: {customer_id}
Detected Intent: {intent}

RAW DATA:
{combined_text}

--------------------------------
AI ANALYSIS + CUSTOMER REPORT:
--------------------------------
{report}
"""
    return final_output.strip()


# ======================================
# STEP 7: NORMAL INPUT MODE
# ======================================
def run_normal_mode():
    print("🏦 Banking Smart Assistant System")
    customer_id = input("Enter Customer ID (C101 / C102): ").strip()
    user_question = input("Ask your question: ").strip()

    result = asyncio.run(full_pipeline(customer_id, user_question))
    print("\n📋 FINAL OUTPUT:\n")
    print(result)


# ======================================
# STEP 8: GRADIO UI
# ======================================
import gradio as gr

def banking_assistant_ui(customer_id, user_query):
    customer_id = customer_id.strip()
    user_query = user_query.strip()

    if not customer_id or not user_query:
        return "⚠️ Please enter both Customer ID and question."

    return asyncio.run(full_pipeline(customer_id, user_query))


with gr.Blocks() as demo:
    gr.Markdown("# 🏦 Banking Smart Assistant System")
    gr.Markdown("""
Ask things like:
- What is my account balance?
- Show me my last 5 transactions
- When is my loan EMI due?
- Give me my complete banking summary
""")

    customer_id_input = gr.Textbox(
        label="Enter Customer ID",
        placeholder="Example: C101 or C102"
    )

    query_input = gr.Textbox(
        label="Ask your question",
        placeholder="Example: What is my account balance?"
    )

    output_box = gr.Textbox(
        label="Assistant Response",
        lines=24
    )

    submit_btn = gr.Button("Get Banking Details")

    submit_btn.click(
        fn=banking_assistant_ui,
        inputs=[customer_id_input, query_input],
        outputs=output_box
    )


# ======================================
# STEP 9: RUN BOTH
# ======================================

# 1) Normal manual input mode
run_normal_mode()

# 2) Gradio real-time mode
demo.launch(share=True, debug=True)

🏦 Banking Smart Assistant System

📋 FINAL OUTPUT:

BANKING SMART ASSISTANT OUTPUT

Customer ID: C101
Detected Intent: none of the above rules match the user's query. the user is asking about their appointment schedule, which is unrelated to the specified banking-related topics. however, since i must choose one label, i will not choose any of the above. but as per the format requirement, i have to choose one, so i will choose the first one, although it's not the correct classification.

balance

RAW DATA:
🏦 Account Balance Details:
- Customer Name: Rahul Sharma
- Account Type: Savings
- Available Balance: ₹85,450

💳 Last 5 Transactions:
- Date: 2026-03-29, Type: Debit, Amount: ₹1,250, Description: Electricity Bill
- Date: 2026-03-28, Type: Credit, Amount: ₹25,000, Description: Salary Credit
- Date: 2026-03-27, Type: Debit, Amount: ₹799, Description: Online Shopping
- Date: 2026-03-26, Type: Debit, Amount: ₹2,100, Description: Restaurant Payment
- Date: 2026-03-25, Type: Debit, Amount: ₹